# Fine-tuning CLIP on ROCOv2 (Medical Imaging)

In [ ]:
# setup environment

# CUDA + PyTorch + CLIP
!pip install torch torchvision --quiet
!pip install git+https://github.com/openai/CLIP.git --quiet
!pip install datasets --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from google.colab import drive, files
import torch
import torch.nn.functional as F
import clip
from torch.utils.data import DataLoader
from tqdm import tqdm
import math

In [3]:
# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Load dataset
dataset = load_dataset("eltorio/ROCOv2-radiology")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

data/train-00000-of-00027.parquet:   0%|          | 0.00/497M [00:00<?, ?B/s]

...

data/test-00005-of-00006.parquet:   0%|          | 0.00/423M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/59962 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/9904 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9927 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/27 [00:00<?, ?it/s]

In [39]:
dataset["train"]

Dataset({
    features: ['image', 'image_id', 'caption', 'cui'],
    num_rows: 59962
})

In [40]:
# Setup device and model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
model = model.float()

print(f"Using device: {device}")

Using device: cuda


In [41]:
# DELETE this section:
# def preprocess_example(example):
#     ...
# small_train = small_train.map(preprocess_example, ...)

# REPLACE WITH this:
from torch.utils.data import Dataset

class CLIPMedicalDataset(Dataset):
    def __init__(self, hf_dataset, preprocess_fn):
        self.dataset = hf_dataset
        self.preprocess = preprocess_fn

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]

        # Get image and convert grayscale to RGB
        img = item["image"]
        if img.mode != "RGB":
            img = img.convert("RGB")

        # Preprocess image (returns tensor)
        pixel_values = self.preprocess(img)

        # Tokenize caption (returns tensor)
        input_ids = clip.tokenize([item["caption"]], truncate=True)[0]

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids
        }

# Create small training subset
small_train_hf = dataset["train"].shuffle(seed=42).select(range(5000))

# Wrap in custom Dataset (CHANGED - no more .map())
train_dataset = CLIPMedicalDataset(small_train_hf, preprocess)

# Then use train_dataset in your DataLoader (change small_train to train_dataset):


In [42]:
# Collate function
# def collate_fn(batch):
#     """Stack batch items into tensors"""
#     return {
#         "images": torch.stack([b["pixel_values"] for b in batch]).to(device, dtype=torch.float32),
#         "input_ids": torch.stack([b["input_ids"] for b in batch]).to(device),
#     }
def collate_fn(batch):
    return {
        "images": torch.stack([b["pixel_values"] for b in batch]).float(),
        "input_ids": torch.stack([b["input_ids"] for b in batch])
    }

# # Create DataLoader
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=8,
#     shuffle=True,
#     collate_fn=collate_fn,
#     num_workers=2
# )

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,   # 🔴 IMPORTANT
    pin_memory=True,
    collate_fn=collate_fn
)


print(f"Training samples: {len(train_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

Training samples: 5000
Batches per epoch: 625


In [43]:
# If you really want a separate parameter, initialize it properly:
logit_scale = torch.nn.Parameter(torch.log(torch.tensor(1 / 0.07)).to(device))  # ~2.66

# This gives initial temperature of ~14, which is reasonable
print(logit_scale)

Parameter containing:
tensor(2.6593, device='cuda:0', requires_grad=True)


In [44]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-6,
    betas=(0.9, 0.98),
    weight_decay=1e-4
)


In [45]:
num_epochs = 3
best_loss = float('inf')

In [46]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=len(train_loader) * num_epochs
)

In [47]:
# Fine-tuning loop

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    num_batches = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, batch in enumerate(progress_bar):
        imgs = batch["images"].to(device)
        txt = batch["input_ids"].to(device)

        if torch.isnan(imgs).any():
           raise RuntimeError("NaN in image tensor")

        # Encode images and text (both trainable now)
        img_features = model.encode_image(imgs)
        txt_features = model.encode_text(txt)

        # Normalize features
        # img_features = F.normalize(img_features, dim=-1, p=2)
        # txt_features = F.normalize(txt_features, dim=-1, p=2) -----------------------
        img_features = img_features / img_features.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        txt_features = txt_features / txt_features.norm(dim=-1, keepdim=True).clamp(min=1e-6)


        # Clamp logit_scale to prevent explosion
        logit_scale.data = torch.clamp(logit_scale.data, max=math.log(100))


        # Compute similarity logits
        logits = logit_scale.exp() * img_features @ txt_features.T

        # Create labels (diagonal matches)
        labels = torch.arange(len(imgs), device=device)

        # Symmetric contrastive loss
        loss_i2t = F.cross_entropy(logits, labels)
        loss_t2i = F.cross_entropy(logits.T, labels)
        loss = (loss_i2t + loss_t2i) / 2

        # Check for NaN
        if torch.isnan(loss) or torch.isinf(loss):
          print(f"\n⚠️ WARNING: NaN/Inf detected at batch {batch_idx}")
          print(f"  logit_scale: {logit_scale.exp().item():.4f}")
          raise RuntimeError("Training failed with NaN/Inf")

        # Backward pass
        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping to prevent explosion
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        # torch.nn.utils.clip_grad_norm_([logit_scale], max_norm=1.0) -----------------------------

        optimizer.step()
        scheduler.step()

        # Track loss
        total_loss += loss.item()
        num_batches += 1

        # Update progress bar
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'temp': f'{logit_scale.exp().item():.2f}',
            'lr': f'{optimizer.param_groups[0]["lr"]:.2e}'
        })

    # Epoch summary
    avg_loss = total_loss / num_batches if num_batches > 0 else float('inf')
    print(f"\nEpoch {epoch+1} - Average Loss: {avg_loss:.4f}, Temperature: {logit_scale.exp().item():.4f}")

    # Save best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'logit_scale': logit_scale,
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }, "/content/drive/MyDrive/clip_med_finetuned_best.pt")
        print(f"Saved best model (loss: {avg_loss:.4f})")

# Save final model
torch.save({
    'model_state_dict': model.state_dict(),
    'logit_scale': logit_scale,
}, "/content/drive/MyDrive/clip_med_finetuned_final.pt")

print("\n✓ Training complete!")

Epoch 1/3: 100%|██████████| 625/625 [03:26<00:00,  3.02it/s, loss=0.6749, temp=14.29, lr=7.50e-07]



Epoch 1 - Average Loss: 0.8502, Temperature: 14.2857
Saved best model (loss: 0.8502)


Epoch 2/3: 100%|██████████| 625/625 [03:21<00:00,  3.10it/s, loss=0.5900, temp=14.29, lr=2.50e-07]



Epoch 2 - Average Loss: 0.4929, Temperature: 14.2857
Saved best model (loss: 0.4929)


Epoch 3/3: 100%|██████████| 625/625 [03:30<00:00,  2.97it/s, loss=0.4589, temp=14.29, lr=0.00e+00]



Epoch 3 - Average Loss: 0.3885, Temperature: 14.2857
Saved best model (loss: 0.3885)

✓ Training complete!


In [49]:
# Validation/Testing
print("\nTesting on validation example...")
model.eval()

test_example = dataset["validation"][0]
test_img = test_example["image"]
if test_img.mode != "RGB":
    test_img = test_img.convert("RGB")
img = preprocess(test_img).unsqueeze(0).to(device, dtype=torch.float32)
captions = ["a chest X-ray", "a PET scan", "an ultrasound image"]
txt = clip.tokenize(captions).to(device)

with torch.no_grad():
    img_feat = model.encode_image(img)
    txt_feat = model.encode_text(txt)
    img_feat = F.normalize(img_feat, dim=-1, p=2)
    txt_feat = F.normalize(txt_feat, dim=-1, p=2)
    scores = (100.0 * img_feat @ txt_feat.T).softmax(dim=-1)

print("\nPrediction scores:")
for cap, score in zip(captions, scores[0]):
    print(f"  {cap}: {score.item():.4f}")

print(f"\nActual caption: {test_example['caption']}")


Testing on validation example...

Prediction scores:
  a chest X-ray: 1.0000
  a PET scan: 0.0000
  an ultrasound image: 0.0000

Actual caption: Chest X-ray showing enlarged cardiac silhouette with cardiothoracic ratio of 70%, and mild pulmonary congestion.
